In [1]:
import torch
import torch.nn as nn
from networks import FullyConnectedNN, NoisyFullyConnectedNN, FixNoiseFullyConnectedNN, DropoutFullyConnectedNN
import os
import pandas as pd
import numpy as np
from utils import *
import shutil
import itertools
from config import configs

{'datasets': ['protein-tertiary-structure'], 'models': ['noisyFCN'], 'optimizer_name': 'Adam', 'hyperparam_eval_interval': 10, 'patience': 3, 'global_max_epochs': 2000, 'dropout_probs': [0.2, 0.1, 0.05, 0.01, 0.005, 0.001], 'fix_noiselevels': [0.001, 0.005, 0.01, 0.05, 0.1], 'init_noiselevels': [0.001, 0.005, 0.01, 0.05, 0.1], 'batch_sizes': [32], 'batch_sizes_specific': {'energy': [32]}, 'weight_decays': [0.01], 'dropout_batch_size': 32, 'n_hidden': 2, 'nonlinearity': 'relu', 'learning_rates': [0.002], 'mc_samples': 500, 'num_units': 50, 'num_units_specific': {'protein-tertiary-structure': 100}, 'normalize_X': True, 'normalize_y': True, 'n_folds': 5, 'inverted_cv_fraction': 5}


In [14]:
model_paths = os.listdir('./save_model')

In [2]:
res = []
dataset_names = configs['datasets']
model_names = configs['models']
learning_rate = configs['learning_rate']
optimizer_name = configs['optimizer_name']
hyperparam_eval_interval = configs['hyperparam_eval_interval']
patience = configs['patience']
global_max_epochs = configs['global_max_epochs']
dropout_probs = configs['dropout_probs']
fix_noiselevels = configs['fix_noiselevels']
init_noiselevels = configs['init_noiselevels']
dropout_batch_size = configs['dropout_batch_size']
n_hidden = configs['n_hidden']
nonlinearity = configs['nonlinearity']
test_times = configs['mc_samples']
normalize_X = configs['normalize_X']
normalize_y = configs['normalize_y']
n_folds = configs['n_folds']
inverted_cv_fraction = configs['inverted_cv_fraction']
use_cuda = torch.cuda.is_available()

KeyError: 'learning_rate'

In [3]:
def normalize(X_train, y_train, normalize_X=True, normalize_y=False):
    X_mean = X_train.mean(axis=0)
    X_std = X_train.std(axis=0)
    y_mean = y_train.mean(axis=0)
    y_std = y_train.std(axis=0)

    # Set std dev to 1 for constant features
    X_std[np.all(X_train == X_train[0, :], axis=0)] = 1.0

    X_train = (X_train - X_mean) / X_std if normalize_X else X_train
    y_train = (y_train - y_mean) / y_std if normalize_y else y_train
    return X_train, y_train

In [14]:
data_loaders = []
models = []
dataset_name = "bostonHousing"
eval_path = os.path.join(os.getcwd(), 'evaluations')
# Create eval dir for dataset
dataset_path = os.path.join(eval_path, dataset_name)
folds_path = os.path.join(dataset_path, 'fold_indices')
feature_indices, target_indices = load_uci_info(dataset_name)
input_size = len(feature_indices)
output_size = len(target_indices)
batch_sizes = configs['batch_sizes_specific'].get('energy',  configs['batch_sizes'])
num_units = configs['num_units_specific'].get('protein-tertiary-structure',  configs['num_units'])
hidden_sizes = [num_units] * n_hidden
model_name = 'FCN'
batch_size = 32
init_noiselevel = 0.01
fix_noiselevel = 0.01
dropout_prob = 0.05
for fold in range(n_folds):
    print('fold {}'.format(fold))
    # 准备好数据集
    X_train, y_train, X_val, y_val = load_fold(folds_path, fold, X_full, y_full)
    X_train, y_train = normalize(X_train, y_train, normalize_X, normalize_y)
    X_val, y_val = normalize(X_val, y_val, normalize_X, normalize_y)
    
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32)
    train_data = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=False, num_workers=4)

    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.float32)
    val_data = torch.utils.data.TensorDataset(X_val, y_val)
    val_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4)

    data_loaders.append((train_loader, val_loader))

    # 创建模型
    input_size = X_train.shape[-1]
    model = def_model(model_name, input_size, hidden_sizes, output_size, init_noiselevel, fix_noiselevel, dropout_prob)
    models.append(model)

    normal_param = [
        param for name, param in model.named_parameters()
        if (not 'alpha_' in name and not 'alphafix_' in name)
    ] # this is the parameters do not contain noise scale coefficient

    alpha_param = [
    param for name, param in model.named_parameters()
    if 'alpha_' in name
    ]



fold 0
model_name FCN
fold 1
model_name FCN
fold 2
model_name FCN
fold 3
model_name FCN
fold 4
model_name FCN


In [5]:
folds_loss_last = []
folds_loss_best = []
for fold in range(n_folds):
    print('fold {}'.format(fold))
    X_train, y_train, X_val, y_val = load_fold(folds_path, fold, X, y)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.float32)
    val_data = torch.utils.data.TensorDataset(X_val, y_val)
    val_loader = torch.utils.data.DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4)
    last_model = def_model(model_name, input_size, hidden_sizes, output_size)
    best_model = def_model(model_name, input_size, hidden_sizes, output_size)
    criterion = nn.MSELoss()
    if use_cuda:
        last_model.cuda()
        best_model.cuda()
        criterion.cuda()

    loss_in_fold_last = []
    loss_in_fold_best = []
    save_path = get_save_path()
    print('save_path: {}'.format(save_path))
    for subdir in os.listdir(save_path):
        model_path = os.path.join(save_path, subdir)
        last_model_path = os.path.join(model_path, 'checkpoint.pth.tar')
        best_model_path = os.path.join(model_path, 'model_best.pth.tar')
        last_checkpoint = torch.load(last_model_path)
        best_checkpoint = torch.load(best_model_path)

        state_tmp = last_model.state_dict()
        if 'state_dict' in last_checkpoint.keys():
            state_tmp.update(last_checkpoint['state_dict'])
        else:
            state_tmp.update(last_checkpoint)

        last_model.load_state_dict(state_tmp)
        print("=> loaded last_model '{}' (epoch {})".format(last_model_path, last_checkpoint['epoch']))

        state_tmp = best_model.state_dict()
        if 'state_dict' in best_checkpoint.keys():
            state_tmp.update(best_checkpoint['state_dict'])
        else:
            state_tmp.update(best_checkpoint)

        best_model.load_state_dict(state_tmp)
        print("=> loaded best_model '{}' (epoch {})".format(best_model_path, best_checkpoint['epoch']))

        val_loss_last, _, _, msll_last = validate(model=last_model, val_loader=val_loader, criterion=criterion)
        val_loss_best, _, _, msll_best = validate(model=best_model, val_loader=val_loader, criterion=criterion)

        loss_in_fold_last.append(val_loss_last)
        loss_in_fold_best.append(val_loss_best)
    
    if len(os.listdir(save_path)) != 5:
        print('number of models is not 5, but {}'.format(len(os.listdir(save_path))))
    folds_loss_last.append(loss_in_fold_last)
    folds_loss_best.append(loss_in_fold_best)

fold 0
model_name FCN
model_name FCN
save_path: /home/xueqiong/noise_injection_uncertainty/regression/evaluations/concrete/save_model/FCN_bs_32_lr_0.005_wd0.0_optm_Adam/fold_0
=> loaded last_model '/home/xueqiong/noise_injection_uncertainty/regression/evaluations/concrete/save_model/FCN_bs_32_lr_0.005_wd0.0_optm_Adam/fold_0/2024-06-17_1729/checkpoint.pth.tar' (epoch 50)
=> loaded best_model '/home/xueqiong/noise_injection_uncertainty/regression/evaluations/concrete/save_model/FCN_bs_32_lr_0.005_wd0.0_optm_Adam/fold_0/2024-06-17_1729/model_best.pth.tar' (epoch 50)
=> loaded last_model '/home/xueqiong/noise_injection_uncertainty/regression/evaluations/concrete/save_model/FCN_bs_32_lr_0.005_wd0.0_optm_Adam/fold_0/2024-06-16_2312/checkpoint.pth.tar' (epoch 50)
=> loaded best_model '/home/xueqiong/noise_injection_uncertainty/regression/evaluations/concrete/save_model/FCN_bs_32_lr_0.005_wd0.0_optm_Adam/fold_0/2024-06-16_2312/model_best.pth.tar' (epoch 39)
=> loaded last_model '/home/xueqiong

In [7]:
folds_loss_last = np.asarray(folds_loss_last)
folds_loss_best = np.asarray(folds_loss_best)
cv_loss_mean_last = np.mean(folds_loss_last)
cv_loss_mean_best = np.mean(folds_loss_best)
cv_loss_std_last = np.std(np.mean(folds_loss_last, axis=0))
cv_loss_std_best = np.std(np.mean(folds_loss_best, axis=0))

In [12]:
dataset_name, cv_loss_mean_last, cv_loss_std_last, cv_loss_mean_best, cv_loss_std_best

('concrete',
 47.92476045301766,
 1.170759948126187,
 41.93276762068721,
 1.3942197866749129)

In [10]:
res = []
res.append([dataset_name, cv_loss_mean_last, cv_loss_std_last, cv_loss_mean_best, cv_loss_std_best])